In [2]:
import pandas as pd

In [3]:
train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")
sample=pd.read_csv("sample_submission.csv")

In [4]:
print("训练集大小：",train.shape)
print("测试集大小:",test.shape)

display(train.head())
train.info()

print("标签比例:")
print(train["Will_Buy_EV"].value_counts(normalize=True))

print("各列缺失数量:")
print(train.isna().sum())

训练集大小： (668665, 15)
测试集大小: (286571, 14)


,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  object 
 9   City_Type                    668665 non-null  object 
 10  Current_Car_Type             668665 non-null  object 
 11  Home_Charging_Possible       668665 non-null  object 
 12  Subsidy_Available            668665 non-null  object 
 13 

<h3 style="color: #3498db;">  Step 1: 确定输入X和标签y  </h3>

In [5]:
X=train.drop(columns=["id","Will_Buy_EV"])
y=train["Will_Buy_EV"].map({
    "No":0,
    "Yes":1
})

print(X.shape)
print(y.value_counts(normalize=True))

(668665, 13)
Will_Buy_EV
0    0.825355
1    0.174645
Name: proportion, dtype: float64


<h3 style="color: #3498db;">  Step 2: 找类别特征  </h3>

In [6]:
cat_cols=X.select_dtypes(
    include=["object"]
).columns.tolist()

print(cat_cols)

['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


<h3 style="color: #3498db;">  Step 3: 划分验证集  </h3>

In [7]:
from sklearn.model_selection import train_test_split

X_train,X_vaild,y_train,y_vaild=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

model=CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    eval_metric="AUC",
    cat_features=cat_cols,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_vaild,y_vaild),
    early_stopping_rounds=100
)

0:	test: 0.9304594	best: 0.9304594 (0)	total: 462ms	remaining: 7m 41s
100:	test: 0.9398433	best: 0.9398433 (100)	total: 20s	remaining: 2m 57s
200:	test: 0.9403294	best: 0.9403294 (200)	total: 42.3s	remaining: 2m 48s
300:	test: 0.9406927	best: 0.9406927 (300)	total: 1m 5s	remaining: 2m 32s
400:	test: 0.9409753	best: 0.9409755 (399)	total: 1m 30s	remaining: 2m 14s
500:	test: 0.9411352	best: 0.9411363 (497)	total: 1m 55s	remaining: 1m 55s
600:	test: 0.9412655	best: 0.9412655 (600)	total: 2m 20s	remaining: 1m 33s
700:	test: 0.9413086	best: 0.9413111 (697)	total: 2m 47s	remaining: 1m 11s
800:	test: 0.9413495	best: 0.9413522 (792)	total: 3m 14s	remaining: 48.3s
900:	test: 0.9413454	best: 0.9413595 (842)	total: 3m 41s	remaining: 24.3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9413594847
bestIteration = 842

Shrink model to first 843 iterations.


CatBoostClassifier(cat_features=['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level'], depth=8, eval_metric='AUC', iterations=1000, learning_rate=0.05, verbose=100)

In [9]:
print(model.best_iteration_)

842


In [10]:
pred = model.predict_proba(X_vaild)[:,1]

print(
    roc_auc_score(y_vaild,pred)
)

0.9413594846833798


In [11]:
X_test=test.drop(columns=["id"])
test_pred=model.predict_proba(X_test)[:,1]

In [12]:
sample.head()

,id,Will_Buy_EV
0,668665,0.174645
1,668666,0.174645
2,668667,0.174645
3,668668,0.174645
4,668669,0.174645


In [13]:
submission=sample.copy()
submission["Will_Buy_EV"]=test_pred
submission.head()

,id,Will_Buy_EV
0,668665,0.010764
1,668666,0.020420
2,668667,0.003990
3,668668,0.001837
4,668669,0.022663


In [14]:
submission.to_csv(
    "submission.csv",
    index=False
)

In [15]:
importance = model.get_feature_importance()

print(
    sorted(
        zip(X.columns, importance),
        key=lambda x:x[1],
        reverse=True
    )
)

[('Subsidy_Available', 49.833037857965664), ('Environmental_Concern_Level', 18.361717123119938), ('Annual_Income_USD', 10.434651598005017), ('Age', 3.9803976452048944), ('Daily_Commute_km', 3.832376535557212), ('Range_Anxiety_Level', 3.519846641913553), ('Charging_Stations_Near_Home', 2.7060769370400566), ('Charging_Stations_Near_Work', 2.3338030167551347), ('Number_of_Cars_Owned', 1.438913267010915), ('Current_Car_Type', 1.3857675898178439), ('Home_Charging_Possible', 1.0761562646760663), ('City_Type', 0.6209550109035863), ('Gender', 0.4763005120301365)]


In [16]:
model.get_feature_importance(
    type="PredictionValuesChange"
)

array([ 3.98039765, 10.4346516 ,  3.83237654,  1.43891327,  2.70607694,
        2.33380302, 18.36171712,  0.47630051,  0.62095501,  1.38576759,
        1.07615626, 49.83303786,  3.51984664])